# Benchmark for Proof of Thought

[Proof of Thought](https://debarghag.github.io/proofofthought/) is a neurosymbolic program for solving true/false natural language reasoning questions. Its pipeline is

```Question from Dataset --> LLM Translation -->  Formal Program (SMT or JSON-based) --> Z3 Execution --> SAT/UNSAT Answer --> Boolean Answer```

The LLM gets three attempts to produce a formalization that returns SAT or UNSAT rather than an error or no result from Z3.

The purpose of this notebook is to provide a simple interface for running a benchmark for Proof of Thought via either Azure OpenAI API or the regular OpenAI API.

For the notebook to work, it requires a cpu environment such as the Conda environment shown below:
```
name: pythonhpcpu
channels:
  - conda-forge
dependencies:
  - python=3.12
  - jupyterlab
  - numba
  - pandas
  - matplotlib
  - h5py
  - ipywidgets
  - numpy
  - scipy
  - scikit-learn
  - pytables
  - xarray
  - netcdf4
  - jupyter-resource-usage
  - transformers
  - pytorch
  - datasets
  - peft
  - pip
```

Steps:
1. Setup and imports.
2. Choose backend (SMT2 or JSON) and LLM configuration (Azure OpenAI or regular OpenAI).
3. Preprocess the dataset, then create the evaluation pipeline.
4. Run the evaluation pipeline, storing all output in a file.

In [1]:
import json
import logging
import sys
import os
from pathlib import Path
from typing import Literal, Any

# Basically the IPython Magic version of these Linux commands for installations
# cd ..
# pip install -r requirements.txt
# pip install -e .
path_words = os.getcwd().split("/")
if path_words[len(path_words)-1] == "benchmark":
    %cd ..
%pip install -r requirements.txt
%pip install -e .

# Add parent directory to path for z3adapter imports
sys.path.insert(0, str(Path(os.getcwd()).parent.parent))

from utils.azure_config import get_client_config
from z3adapter.reasoning import EvaluationPipeline, ProofOfThought
from openai import OpenAI

# Configure logging (output messages during the benchmark). This stores all logs in the outputLog.txt file.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("outputLog.txt", mode="a"),
        logging.StreamHandler(),
    ],
)

/home/kyi/proofofthought
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached anyio-4.11.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached babel-2.17.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached backrefs-5.9-py312-none-any.whl.metadata (3.2 kB)
  Using cached black-25.9.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_28_x86_64.whl.metadata (83 kB)
  Using cached certifi-2025.8.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached cfgv-3.4.0-py2.py3-none-any.whl.metadata (8.5 kB)
  Using cached charset_normalizer-3.4.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (37 kB)
  Using cached click-8.3.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached distlib-0.4.0-py2.py3-none-any.whl.metadata (5.2 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached ghp_import-2.1.0-py3-none-any.whl.metadata 

In [3]:
# Replace the ____ at the end with either "smt2" or "json"
BACKEND: Literal["json", "smt2"] = "smt2"
# Replace the ____ at the end with either "prontoqa" or "folio" or "proofwriter" or "conditionalqa" or "strategyqa"
DATASET: Literal["prontoqa", "folio", "proofwriter", "conditionalqa", "strategyqa"] = "proofwriter"


# THE FOLLOWING LINES OF CODE APPLY FOR REGULAR OPENAI API. COMMENT THESE OUT IF YOU ARE USING AZURE OPENAI.
# Replace value with OpenAI API key. This command stores the key as an environment variable.
%env OPENAI_API_KEY redacted

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set. " "Please set it in your .env file or environment variables."
    )
client = OpenAI(
    api_key=api_key,
    base_url="https://ellm.nrp-nautilus.io/v1", # Replace with "https://ellm.nrp-nautilus.io/v1" if using NRP's LLMs.
    # Replace with "https://api.openai.com/v1" if using OpenAI API Platform
)
pot = ProofOfThought(
    llm_client=client,
    model="gpt-oss", # Read the API documentation to get the model name. For example, put "gpt-5" for OpenAI Platform's GPT-5 model.
    backend=BACKEND,
    max_attempts=3,
    cache_dir=f"output/{BACKEND}_programs_{DATASET}",
    z3_path="z3",
)


# THE FOLLOWING LINES OF CODE APPLY FOR AZURE OPENAI API. COMMENT THESE OUT IF YOU ARE USING REGULAR OPENAI.
# # Get Azure OpenAI configuration. To change it, go to proofofthought/utils/azure_config.py
# config = get_client_config()

# # Create ProofOfThought instance with configurable backend
# pot = ProofOfThought(
#     llm_client=config["llm_client"],
#     model=config["model"],
#     backend=BACKEND,
#     max_attempts=3,
#     cache_dir=f"output/{BACKEND}_programs_{DATASET}",
#     z3_path="z3",
# )

env: OPENAI_API_KEY=redacted


In [4]:
DATASET_PATH = "" # dataset's file path for the evaluation pipeline. defined below, no need for user input.

SKIP_PREPROCESS = True # SET THIS: If you have already run a benchmark using the current dataset, then the dataset is already preprocessed, so set to True.
# This only affects the ProntoQA, FOLIO, and ProofWriter datasets, because the ConditionalQA dataset preprocessing can detect whether the dataset was already preprocessed, and the StrategyQA dataset doesn't need to be preprocessed.

# preprocessing the specified dataset. as there are five possible datasets, the preprocessing code is quite long
if DATASET == "prontoqa":
    def preprocess_prontoqa(input_path: str, output_path: str) -> None:
        """Preprocess ProntoQA to combine context + question and convert answer format.
    
        ProntoQA format:
        - context: Background facts and rules
        - question: "Is the following statement true or false? [statement]"
        - answer: "A" (True) or "B" (False)
    
        Output format:
        - question: context + "\n\n" + question
        - answer: True/False (boolean)
        - id: original id
        """
        with open(input_path) as f:
            data = json.load(f)
    
        processed = []
        for item in data:
            # Combine context with question for complete reasoning scenario
            full_question = f"{item['context']}\n\n{item['question']}"
    
            # Convert A/B answer to boolean (A=True, B=False)
            answer_bool = True if item["answer"] == "A" else False
    
            processed.append(
                {
                    "id": item["id"],
                    "question": full_question,
                    "answer": answer_bool,
                    "original_context": item["context"],
                    "original_question": item["question"],
                }
            )
    
        with open(output_path, "w") as f:
            json.dump(processed, f, indent=2)
    
        print(f"Preprocessed {len(processed)} ProntoQA examples")
        print(f"Saved to: {output_path}")
    
    
    # Preprocess the dataset
    input_file = "data/ProntoQA_dev_gpt-4.json"
    processed_file = "data/ProntoQA_dev_processed.json"

    if (SKIP_PREPROCESS == False):
        print("Preprocessing ProntoQA dataset...")
        preprocess_prontoqa(input_file, processed_file)
        print()
    else:
        print("Skipping Preprocess for ProntoQA dataset...")
        print()
    DATASET_PATH = "data/ProntoQA_dev_processed.json"
elif DATASET == "folio":
    def preprocess_folio(jsonl_path: str, output_path: str) -> None:
        """Preprocess FOLIO to combine premises + conclusion and filter Uncertain labels.
    
        FOLIO format:
        - premises: Background context (natural language statements)
        - conclusion: The statement to verify
        - label: "True", "False", or "Uncertain"
        - story_id: ID of the premise set
        - example_id: Unique example ID
    
        Output format:
        - question: premises + "\n\nConclusion: " + conclusion
        - answer: True/False (boolean, "Uncertain" samples filtered out)
        - id: example_id
        - story_id: original story_id
        """
        with open(jsonl_path) as f:
            data = [json.loads(line) for line in f]
    
        print(f"Total samples: {len(data)}")
    
        # Filter out Uncertain labels for binary classification
        data_filtered = [item for item in data if item["label"] != "Uncertain"]
    
        print(f"After filtering Uncertain: {len(data_filtered)}")
    
        label_counts: dict[str, int] = {}
        for item in data_filtered:
            label_counts[item["label"]] = label_counts.get(item["label"], 0) + 1
        print(f"Label distribution: {label_counts}")
    
        processed = []
        for item in data_filtered:
            # Combine premises with conclusion for complete reasoning scenario
            # Frame as direct assertion task to match SAT=True/UNSAT=False interpretation
            # Instead of asking "does it follow?", ask "is it true given the premises?"
            full_question = f"Given the following premises:\n\n{item['premises']}\n\nAssuming all premises are true, is the following statement true or false?\n\nStatement: {item['conclusion']}"
    
            # Convert string label to boolean
            answer_bool = item["label"] == "True"
    
            processed.append(
                {
                    "id": f"folio_{item['example_id']}",
                    "question": full_question,
                    "answer": answer_bool,
                    "story_id": item["story_id"],
                    "example_id": item["example_id"],
                    "original_premises": item["premises"],
                    "original_conclusion": item["conclusion"],
                }
            )
    
        with open(output_path, "w") as f:
            json.dump(processed, f, indent=2)
    
        print(f"Preprocessed {len(processed)} FOLIO examples")
        print(f"Saved to: {output_path}")
    
    
    # Preprocess the dataset
    jsonl_file = "data/folio_v2_train.jsonl"
    processed_file = "data/folio_v2_train_processed.json"

    if (SKIP_PREPROCESS == False):
        print("Preprocessing FOLIO dataset...")
        preprocess_folio(jsonl_file, processed_file)
        print()
    else:
        print("Skipping preprocess for FOLIO dataset...")
        print()
    DATASET_PATH = "data/folio_v2_train_processed.json"
elif DATASET == "proofwriter":
    def preprocess_proofwriter(parquet_path: str, output_path: str, max_samples: int = 1000) -> None:
        """Preprocess ProofWriter to combine theory + question and filter Unknown answers.
    
        ProofWriter format:
        - theory: Background facts and rules (natural language)
        - question: The query statement
        - answer: "True", "False", or "Unknown"
        - QDep: Question depth (number of reasoning steps)
    
        Output format:
        - question: theory + "\n\nQuestion: " + question
        - answer: True/False (boolean, "Unknown" samples filtered out)
        - id: original id
        - QDep: reasoning depth
        """
        import pandas as pd
    
        df = pd.read_parquet(parquet_path)
    
        # Filter out Unknown answers for binary classification
        df_filtered = df[df["answer"] != "Unknown"].copy().reset_index(drop=True)

        # Make ids unique, since for each context, there are a few questions with the same id.
        used_ids = {}
        for i in range(len(df_filtered)):
            # The first time it sees the id, add the id to used_ids (which is a dictionary) with the number 1.
            # The second time it sees the id, replace 1 in the dictionary with 2, then add "(1)" to the id.
            # The third time it sees the id, replace 2 in the dictionary with 3, then add "(2)" to the id. And so on...
            if df_filtered.at[i,"id"] in used_ids.keys():
                used_ids[df_filtered.at[i,"id"]] += 1
                df_filtered.at[i,"id"] += f"({used_ids[df_filtered.at[i,"id"]]-1})"
            else:
                used_ids[df_filtered.at[i,"id"]] = 1

        print(f"First few elements: {df_filtered.head()}")
        print(f"Total samples: {len(df)}")
        print(f"After filtering Unknown: {len(df_filtered)}")
        print(f"Answer distribution: {df_filtered['answer'].value_counts().to_dict()}")
    
        # Sample for evaluation (balanced if possible)
        if len(df_filtered) > max_samples:
            # Try to get balanced sample
            df_true = df_filtered[df_filtered["answer"] == "True"].sample(
                n=min(max_samples // 2, len(df_filtered[df_filtered["answer"] == "True"])),
                random_state=42,
            )
            df_false = df_filtered[df_filtered["answer"] == "False"].sample(
                n=min((max_samples+1) // 2, len(df_filtered[df_filtered["answer"] == "False"])),
                random_state=42,
            )
            df_filtered = pd.concat([df_true, df_false]).sample(frac=1, random_state=42)
    
        processed = []
        for _, row in df_filtered.iterrows():
            # Combine theory with question for complete reasoning scenario
            full_question = f"{row['theory']}\n\nQuestion: {row['question']}"
    
            # Convert string answer to boolean
            answer_bool = row["answer"] == "True"
    
            processed.append(
                {
                    "id": row["id"],
                    "question": full_question,
                    "answer": answer_bool,
                    "QDep": int(row["QDep"]),
                    "original_theory": row["theory"],
                    "original_question": row["question"],
                }
            )
    
        with open(output_path, "w") as f:
            json.dump(processed, f, indent=2)
    
        print(f"Preprocessed {len(processed)} ProofWriter examples")
        print(f"QDep distribution: {df_filtered['QDep'].value_counts().sort_index().to_dict()}")
        print(f"Saved to: {output_path}")
    
    
    # Preprocess the dataset
    parquet_file = "data/proofwriter_test.parquet"
    processed_file = "data/proofwriter_test_processed.json"

    if (SKIP_PREPROCESS == False):
        print("Preprocessing ProofWriter dataset...")
        preprocess_proofwriter(parquet_file, processed_file, max_samples=10000)
        print()
    else:
        print("Skipping preprocess for ProofWriter dataset...")
        print()
    DATASET_PATH = "data/proofwriter_test_processed.json"
elif DATASET == "conditionalqa":
    def clean_html_content(html_list: list[str]) -> str:
        """Convert list of HTML snippets to clean text for reasoning.
    
        Args:
            html_list: List of HTML content strings
    
        Returns:
            Clean text with basic structure preserved
        """
        import re
    
        text_parts = []
        for html in html_list:
            # Remove HTML tags but preserve structure
            text = re.sub(r"<h[1-6]>", "\n## ", html)
            text = re.sub(r"</h[1-6]>", "\n", text)
            text = re.sub(r"<p>", "\n", text)
            text = re.sub(r"</p>", "\n", text)
            text = re.sub(r"<li>", "\n- ", text)
            text = re.sub(r"</li>", "", text)
            text = re.sub(r"<tr>", "\n| ", text)
            text = re.sub(r"</tr>", " |", text)
            text = re.sub(r"<[^>]+>", "", text)
    
            # Clean up whitespace
            text = re.sub(r"\n\s*\n\s*\n", "\n\n", text)
            text = text.strip()
    
            if text:
                text_parts.append(text)
    
        return "\n\n".join(text_parts)
    
    
    def preprocess_documents(documents_path: str, output_path: str) -> dict[str, dict[str, Any]]:
        """Preprocess ConditionalQA documents for reasoning.
    
        Args:
            documents_path: Path to conditionalqa_documents.json
            output_path: Path to save preprocessed documents
    
        Returns:
            Dictionary mapping URL to processed document data
        """
        with open(documents_path) as f:
            documents = json.load(f)
    
        print(f"Total documents: {len(documents)}")
    
        processed_docs = {}
        for doc in documents:
            url = doc["url"]
            title = doc["title"]
            contents_text = clean_html_content(doc["contents"])
    
            processed_docs[url] = {
                "url": url,
                "title": title,
                "content": contents_text,
                "original_contents": doc["contents"],
            }
    
        with open(output_path, "w") as f:
            json.dump(processed_docs, f, indent=2)
    
        print(f"Preprocessed {len(processed_docs)} documents")
        print(f"Saved to: {output_path}")
    
        return processed_docs
    
    
    def preprocess_questions(
        questions_path: str,
        documents: dict[str, dict[str, Any]],
        output_path: str,
        max_samples: int | None = None,
    ) -> list[dict[str, Any]]:
        """Preprocess ConditionalQA questions for evaluation.
    
        ConditionalQA format:
        - url: Link to source document
        - scenario: User's situation/context
        - question: Specific question about the scenario
        - not_answerable: Whether question can be answered from document
        - answers: List of [answer_text, supporting_facts]
        - evidences: List of HTML snippets supporting the answer
    
        Output format:
        - id: Original question ID
        - question: scenario + question + document content (MINIMAL CHANGES)
        - answer: Boolean (True/False) for yes/no answers only
        - url: Document URL
        - answerable: Whether question is answerable
    
        Note: Only includes questions with boolean yes/no answers.
        """
        with open(questions_path) as f:
            questions = json.load(f)
    
        print(f"Total questions: {len(questions)}")
    
        processed = []
        answerable_count = 0
        not_answerable_count = 0
        skipped_non_boolean = 0
    
        for q in questions:
            url = q["url"]
    
            # Get associated document
            if url not in documents:
                print(f"Warning: No document found for URL {url}")
                continue
    
            doc = documents[url]
    
            # Parse answer - ConditionalQA answers are [answer_text, supporting_facts]
            answer_text = q["answers"][0][0] if q["answers"] else None
            if not answer_text:
                continue
    
            # Convert to boolean (filter to boolean-only)
            answer_lower = answer_text.lower().strip()
            if answer_lower in ["yes", "true"]:
                answer_bool = True
            elif answer_lower in ["no", "false"]:
                answer_bool = False
            else:
                # Skip non-boolean answers
                skipped_non_boolean += 1
                continue
    
            # Combine scenario, question, and document (MINIMAL - no transformation)
            full_question = (
                f"Scenario: {q['scenario']}\n\n"
                f"Question: {q['question']}\n\n"
                f"Document - {doc['title']}:\n{doc['content']}"
            )
    
            is_answerable = not q.get("not_answerable", False)
            if is_answerable:
                answerable_count += 1
            else:
                not_answerable_count += 1
    
            processed.append(
                {
                    "id": q["id"],
                    "question": full_question,
                    "answer": answer_bool,
                    "answer_text": answer_text,
                    "url": url,
                    "answerable": is_answerable,
                    "scenario": q["scenario"],
                    "original_question": q["question"],
                }
            )
    
            if max_samples and len(processed) >= max_samples:
                break
    
        with open(output_path, "w") as f:
            json.dump(processed, f, indent=2)
    
        print(f"Preprocessed {len(processed)} questions (boolean answers only)")
        print(f"  - Answerable: {answerable_count}, Not answerable: {not_answerable_count}")
        print(f"  - Skipped non-boolean: {skipped_non_boolean}")
        print(f"Saved to: {output_path}")
    
        return processed
    
    documents_file = "data/conditionalqa_documents.json"
    processed_docs_file = "data/conditionalqa_documents_processed.json"

    print("=" * 70)
    print("STAGE 1: PREPROCESSING DOCUMENTS")
    print("=" * 70)

    if os.path.exists(processed_docs_file):
        print(f"Loading preprocessed documents from {processed_docs_file}")
        with open(processed_docs_file) as f:
            processed_docs = json.load(f)
    else:
        print("Preprocessing ConditionalQA documents...")
        processed_docs = preprocess_documents(documents_file, processed_docs_file)

    print()

    # Preprocess questions
    questions_file = "data/conditionalqa_train.json"
    processed_questions_file = "data/conditionalqa_train_processed.json"

    print("=" * 70)
    print("STAGE 2: PREPROCESSING QUESTIONS")
    print("=" * 70)

    if os.path.exists(processed_questions_file):
        print(f"Loading preprocessed questions from {processed_questions_file}")
    else:
        print("Preprocessing ConditionalQA questions...")
        preprocess_questions(questions_file, processed_docs, processed_questions_file)

    print()
    DATASET_PATH = "data/conditionalqa_train_processed.json"
elif DATASET == "strategyqa":
    DATASET_PATH = "data/strategyQA_train.json"
else:
    raise ValueError(
        f"Dataset value is: {DATASET}. This value is invalid."
    )

# Create evaluation pipeline with parallel workers
num_workers=10
evaluator = EvaluationPipeline(
    proof_of_thought=pot,
    output_dir=f"output/{BACKEND}_evaluation_{DATASET}",
    num_workers=num_workers,
)
print(f"Evaluation pipeline created with {num_workers} workers in parallel.")

Skipping preprocess for ProofWriter dataset...

Evaluation pipeline created with 10 workers in parallel.


In [5]:
# IF THE output FOLDER EXISTS BEFORE YOU RUN THIS CELL,
# note that the model listed at the end of this cell's output is the one you specified when initializing the object pot,
# not necessarily the model used to create the cached data.

# Run the evaluation pipeline
result = evaluator.evaluate(
    dataset=processed_file,
    question_field="question",
    answer_field="answer",
    id_field="id",
    max_samples=10, # 10 for debug
    skip_existing=True,
    # For better output of final evaluation results
    dataset_name=DATASET,
)

2026-07-18 21:49:31,003 - INFO - Evaluating 10 samples with 10 workers
2026-07-18 21:49:31,004 - INFO - Using parallel processing with threading
2026-07-18 21:49:31,005 - INFO - [1/10] Processing: RelNeg-OWA-D1-2643(11)
2026-07-18 21:49:31,005 - INFO - [2/10] Processing: RelNeg-OWA-D5-338
2026-07-18 21:49:31,005 - INFO - [3/10] Processing: AttNonegNatLang-OWA-132(6)
2026-07-18 21:49:31,006 - INFO - Processing question: The bald eagle is not big. The bald eagle is blue. The bald eagle likes the mouse. The cow chases the bald eagle. The cow chases the mouse. The cow is round. The cow likes the bald eagle. The cow needs the bald eagle. The cow does not need the mouse. The mouse chases the cow. The mouse is big. The mouse is green. The mouse does not like the bald eagle. The mouse likes the cow. The mouse needs the cow. If something likes the bald eagle and it chases the bald eagle then the bald eagle does not like the cow. If something needs the mouse then it does not like the mouse. If s

Congratulations! You have completed the benchmark!

If you are going to run the benchmark again on a different LLM, please put the outputLog.txt file in the output folder and then rename the output folder. This will allow the benchmark to actually run for the new LLM rather than use the cached results.